In [1]:
from pathlib import Path
import numpy as np
from urllib.request import urlretrieve
import pandas as pd

In [ ]:
# SNIPS_DATA_BASE_URL = (
#     "https://github.com/ogrisel/slot_filling_and_intent_detection_of_SLU/blob/"
#     "master/data/snips/"
# )
# for filename in ["train", "valid", "test", "vocab.intent", "vocab.slot"]:
#     path = Path(filename)
#     if not path.exists():
#         print(f"Downloading {filename}...")
#         urlretrieve(SNIPS_DATA_BASE_URL + filename + "?raw=true", path)

In [2]:
lines_train = Path('dataset/train').read_text('utf-8').strip().splitlines()
print(f"First line of train dataset: {lines_train[0]}")

First line of train dataset: Add:O Don:B-entity_name and:I-entity_name Sherri:I-entity_name to:O my:B-playlist_owner Meditate:B-playlist to:I-playlist Sounds:I-playlist of:I-playlist Nature:I-playlist playlist:O <=> AddToPlaylist


In [3]:
def parse_line(line):
    utterance_data, intent_label = line.split(" <=> ")
    items = utterance_data.split()
    words = [item.rsplit(':', 1)[0] for item in items]
    word_labels = [item.rsplit(':', 1)[1] for item in items]
    return {
        'intent_label': intent_label,
        'words': " ".join(words),
        'words_label': " ".join(word_labels),
        'length': len(words)
    }
parse_line(lines_train[0])

{'intent_label': 'AddToPlaylist',
 'words': 'Add Don and Sherri to my Meditate to Sounds of Nature playlist',
 'words_label': 'O B-entity_name I-entity_name I-entity_name O B-playlist_owner B-playlist I-playlist I-playlist I-playlist I-playlist O',
 'length': 12}

In [4]:
data = [parse_line(line) for line in lines_train]

In [6]:
data[:5]

[{'intent_label': 'AddToPlaylist',
  'words': 'Add Don and Sherri to my Meditate to Sounds of Nature playlist',
  'words_label': 'O B-entity_name I-entity_name I-entity_name O B-playlist_owner B-playlist I-playlist I-playlist I-playlist I-playlist O',
  'length': 12},
 {'intent_label': 'AddToPlaylist',
  'words': 'put United Abominations onto my rare groove playlist',
  'words_label': 'O B-entity_name I-entity_name O B-playlist_owner B-playlist I-playlist O',
  'length': 8},
 {'intent_label': 'AddToPlaylist',
  'words': 'add the tune by misato watanabe to the Trapeo playlist',
  'words_label': 'O O B-music_item O B-artist I-artist O O B-playlist O',
  'length': 10},
 {'intent_label': 'AddToPlaylist',
  'words': 'add this artist to my this is miguel bosé playlist',
  'words_label': 'O O B-music_item O B-playlist_owner B-playlist I-playlist I-playlist I-playlist O',
  'length': 10},
 {'intent_label': 'AddToPlaylist',
  'words': 'add heresy and the hotel choir to the evening acoustic play

In [8]:
from transformers import BertTokenizer

model_name = 'bert-base-cased'
tokenizer = BertTokenizer.from_pretrained(model_name)

In [16]:
from tqdm import tqdm
def encode(data):
    input_ids = []
    attention_masks = []
    token_type_ids = []
    intent_labels = []
    slot_labels = []
    
    for item in tqdm(data):
        encoding = tokenizer(
            item['words'],
            padding='max_length',
            truncation=True,
            max_length=50,
            return_tensors='np'
        )
        input_ids.append(encoding['input_ids'][0])
        attention_masks.append(encoding['attention_mask'][0])
        token_type_ids.append(encoding['token_type_ids'][0])
        intent_labels.append(item['intent_label'])
        slot_labels.append(item['words_label'])
    
    return {
        'input_ids': np.array(input_ids),
        'attention_masks': np.array(attention_masks),
        'token_type_ids': np.array(token_type_ids),
        'intent_labels': np.array(intent_labels),
        'slot_labels': np.array(slot_labels)
    }

In [17]:
vectors = encode(data)

100%|██████████| 13084/13084 [00:03<00:00, 3938.13it/s]


In [24]:
vectors['input_ids'][1]
#print vocab size
#print(f"Vocab size: {tokenizer.vocab_size}")

array([  101,  1508,  1244,   138,  4043,  9204,  1116,  2135,  1139,
        4054, 22482,  1505,  7276,   102,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0])

In [23]:
intent_names = Path('dataset/vocab.intent').read_text('utf-8').split()
intent_map = dict((label, idx) for idx, label in enumerate(intent_names))
intent_map

{'AddToPlaylist': 0,
 'BookRestaurant': 1,
 'GetWeather': 2,
 'PlayMusic': 3,
 'RateBook': 4,
 'SearchCreativeWork': 5,
 'SearchScreeningEvent': 6}